## Finetune tell-tale classifier (3 classes)

This notebook mirrors the detector Colab flow (`finetune_rtdetr.ipynb`):

1. Download the **3-class** fused YOLO dataset `telltale_3_classes_fused.zip` from [estefoucher/sail-cv-telltales](https://huggingface.co/datasets/estefoucher/sail-cv-telltales).
2. Upload your **custom split** zip (output of `split_yolo_by_source.py` with `--no-reduce`). It must have the same `nc` and `names` order as the main dataset.
3. Fuse main + custom, crop boxes to an Ultralytics **classification** layout, train `yolo11n-cls`.

**Enable GPU (Colab):** **Runtime** → **Change runtime type** → **GPU** → **Save**, then run from the top.


In [ ]:
!nvidia-smi


In [ ]:
import os
HOME = os.getcwd()
print(HOME)


## Install dependencies


In [ ]:
!pip install -q ultralytics huggingface_hub pyyaml opencv-python-headless tqdm
from IPython import display
display.clear_output()
print("OK")


## Download and unzip main dataset from Hugging Face

File: `telltale_3_classes_fused.zip` (YOLO layout: `train/`, `val/`, `data.yaml`, `nc: 3`).


In [ ]:
import zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

main_data_dir = Path(HOME) / "main_data"
main_data_dir.mkdir(parents=True, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="estefoucher/sail-cv-telltales",
    filename="telltale_3_classes_fused.zip",
    repo_type="dataset",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(main_data_dir)

contents = [p for p in main_data_dir.iterdir() if p.is_dir()]
candidate = main_data_dir
for child in contents:
    if (child / "train").exists() or (child / "val").exists():
        candidate = child
        break
if not (candidate / "train").exists() and not (candidate / "val").exists():
    raise FileNotFoundError(
        f"Expected train/ or val/ under {main_data_dir}. Contents: {list(main_data_dir.iterdir())}"
    )
main_data_dir = candidate.resolve()
print(f"Main dataset at: {main_data_dir}")
for split in ("train", "val"):
    img_dir = main_data_dir / split / "images"
    n = len(list(img_dir.iterdir())) if img_dir.exists() else 0
    print(f"  {split}/images: {n} items")


## Upload your custom split dataset (Colab)

Upload a zip of the output of:

`uv run python finetuning/split_yolo_by_source.py EXPORT -o OUT --no-reduce`

The zip must contain `train/`, `val/`, and `data.yaml` with **matching** `names` (and `nc`) to the main HF dataset.

**Also required for the next step:** upload `crop_dataset.py` from this repo (`finetuning/crop_dataset.py`) when prompted, or place it in `HOME` before running the crop cell.


In [ ]:
import zipfile
from pathlib import Path

try:
    from google.colab import files
    print("Upload your custom split zip:")
    up1 = files.upload()
    custom_zip = list(up1.keys())[0]
    custom_data_dir = Path(HOME) / "custom_data"
    custom_data_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(custom_zip, "r") as z:
        z.extractall(custom_data_dir)
    contents = [p for p in custom_data_dir.iterdir() if p.is_dir()]
    for child in contents:
        if (child / "train").exists() or (child / "val").exists():
            custom_data_dir = child.resolve()
            break
    else:
        custom_data_dir = custom_data_dir.resolve()
    print(f"Custom dataset at: {custom_data_dir}")
    print("Upload finetuning/crop_dataset.py from Sail-CV:")
    up2 = files.upload()
    crop_name = list(up2.keys())[0]
    import shutil
    shutil.move(crop_name, str(Path(HOME) / "crop_dataset.py"))
    print("Saved:", Path(HOME) / "crop_dataset.py")
except ImportError:
    custom_data_dir = Path(HOME) / "custom_data"
    print("Not on Colab: set paths manually.")
    print("  custom_data_dir: directory with train/, val/, data.yaml")
    print("  Place crop_dataset.py in HOME or adjust the crop cell.")


## Fuse main + custom (multi-class)


In [ ]:
import shutil
from pathlib import Path

import yaml

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}


def copy_split_fixed(src_base: Path, split: str, dst_img: Path, dst_lbl: Path, prefix: str) -> int:
    img_dir = src_base / split / "images"
    lbl_dir = src_base / split / "labels"
    if not img_dir.exists():
        return 0
    n = 0
    for f in img_dir.iterdir():
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            shutil.copy2(f, dst_img / f"{prefix}_{f.name}")
            n += 1
    if lbl_dir.exists():
        for f in lbl_dir.iterdir():
            if f.is_file() and f.suffix == ".txt":
                shutil.copy2(f, dst_lbl / f"{prefix}_{f.name}")
    return n


def load_split_meta(root: Path) -> tuple[int, list[str]]:
    y = yaml.safe_load((root / "data.yaml").read_text())
    raw = y["names"]
    if isinstance(raw, dict):
        keys = sorted(raw, key=lambda k: int(str(k)))
        names = [str(raw[k]) for k in keys]
    else:
        names = [str(x) for x in raw]
    nc = int(y.get("nc", len(names)))
    return nc, names


main_data_dir = Path(main_data_dir)
custom_data_dir = Path(custom_data_dir)
nc_m, names_m = load_split_meta(main_data_dir)
nc_c, names_c = load_split_meta(custom_data_dir)
if nc_m != nc_c or names_m != names_c:
    raise ValueError(
        f"Main vs custom metadata mismatch.\n"
        f"  main:   nc={nc_m}, names={names_m}\n"
        f"  custom: nc={nc_c}, names={names_c}\n"
        f"Use the same annotator class order as the HF dataset."
    )
nc, names = nc_m, names_m

fused_dir = Path(HOME) / "fused_data"
if fused_dir.exists():
    shutil.rmtree(fused_dir)
fused_dir.mkdir(parents=True, exist_ok=True)
for split in ("train", "val"):
    (fused_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (fused_dir / split / "labels").mkdir(parents=True, exist_ok=True)

print("Main:", main_data_dir)
print("Custom:", custom_data_dir)
for split in ("train", "val"):
    dst_img = fused_dir / split / "images"
    dst_lbl = fused_dir / split / "labels"
    n_main = copy_split_fixed(main_data_dir, split, dst_img, dst_lbl, "main")
    n_custom = 0
    if custom_data_dir.exists() and (custom_data_dir / split).exists():
        n_custom = copy_split_fixed(custom_data_dir, split, dst_img, dst_lbl, "custom")
    print(f"  {split}: {n_main} main, {n_custom} custom images")

quoted = ", ".join(repr(n) for n in names)
fused_yaml = fused_dir / "data.yaml"
fused_yaml.write_text(
    f"path: {fused_dir.resolve()}\n"
    f"train: train/images\n"
    f"val: val/images\n"
    f"nc: {nc}\n"
    f"names: [{quoted}]\n"
)
print(f"Fused YOLO dataset: {fused_dir}")
print(f"data.yaml: nc={nc}, names={names}")


## Load `prepare_balanced_dataset` from `crop_dataset.py`

Expects `crop_dataset.py` in `HOME` (uploaded in the custom-data cell on Colab).


In [ ]:
import importlib.util
from pathlib import Path

crop_py = Path(HOME) / "crop_dataset.py"
if not crop_py.is_file():
    raise FileNotFoundError(
        f"Missing {crop_py}. Upload finetuning/crop_dataset.py from the Sail-CV repo "
        "or copy it to the Colab working directory."
    )
spec = importlib.util.spec_from_file_location("crop_dataset", crop_py)
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mod)
prepare_balanced_dataset = mod.prepare_balanced_dataset
print("Loaded prepare_balanced_dataset from crop_dataset.py")


## Build classification crops (`cls_dataset/`)

Ultralytics expects `cls_dataset/train/<class_id>/*.jpg` (and `val/`).

**Train:** multipliers are computed from **bbox counts** in `fused_data/train/labels` so each class is oversampled toward the majority count (`ceil(max_count / count_c)`), capped at `BALANCE_MAX_MULT` (default 25) to avoid huge folders.

**Val:** multiplier 1 for every class (realistic evaluation distribution).


In [ ]:
import math
import shutil
from collections import Counter
from pathlib import Path

BALANCE_MAX_MULT = 25


def count_boxes_per_class(lbl_dir: Path) -> Counter:
    c: Counter[int] = Counter()
    if not lbl_dir.is_dir():
        return c
    for lf in lbl_dir.glob("*.txt"):
        for raw in lf.read_text().splitlines():
            stripped = raw.strip()
            if not stripped:
                continue
            parts = stripped.split()
            if len(parts) >= 5:
                c[int(float(parts[0]))] += 1
    return c


def train_balance_multipliers(counts: Counter, nc: int, cap: int) -> dict[int, int]:
    """Oversample minority classes so effective targets align with the majority count."""
    per = [counts.get(i, 0) for i in range(nc)]
    positive = [x for x in per if x > 0]
    if not positive:
        return dict.fromkeys(range(nc), 1)
    mx = max(positive)
    out: dict[int, int] = {}
    for i in range(nc):
        ni = per[i]
        if ni <= 0:
            out[i] = 1
        else:
            out[i] = min(cap, max(1, math.ceil(mx / ni)))
    return out


train_lbl = fused_dir / "train" / "labels"
train_counts = count_boxes_per_class(train_lbl)
train_mult = train_balance_multipliers(train_counts, nc, BALANCE_MAX_MULT)

print("Train bbox counts per class:", dict(sorted(train_counts.items())))
print("Train oversampling multipliers:", train_mult)

cls_root = Path(HOME) / "cls_dataset"
if cls_root.exists():
    shutil.rmtree(cls_root)
(cls_root / "train").mkdir(parents=True)
(cls_root / "val").mkdir(parents=True)

prepare_balanced_dataset(
    fused_dir / "train",
    cls_root / "train",
    multipliers=train_mult,
    min_width=10,
    min_height=10,
)
prepare_balanced_dataset(
    fused_dir / "val",
    cls_root / "val",
    multipliers=None,
    min_width=10,
    min_height=10,
)
print(f"Classification dataset at {cls_root}")


## Train classifier


In [ ]:
from ultralytics import YOLO
import os

model = YOLO("yolo11n-cls.pt")
model.train(
    data=str(cls_root),
    epochs=50,
    imgsz=224,
    batch=32,
    degrees=15.0,
    flipud=0.5,
    fliplr=0.5,
    project=os.path.join(HOME, "runs", "classify"),
    name="train",
)
metrics = model.val()
print(f"Top-1 accuracy: {metrics.top1:.4f}")


## Output

Best weights: `runs/classify/train/weights/best.pt`. Download from Colab and set `model_path` in your classifier parameters YAML (or place under `checkpoints/`).

**Dev shortcut:** To train on **custom data only**, skip the HF download cell, set `main_data_dir = custom_data_dir` before fusion, or fuse with zero files from main (not recommended—better to duplicate the crop cell pointing at `custom_data_dir` only).
